In [2]:

import pandas as pd


In [3]:
df = pd.read_csv('fundamentals_with_prices.csv')

df = df.sort_values(['ticker', 'price_date'])

df['target'] = (df['close_price'].shift(-1) > df['close_price']).astype(int)

In [4]:

drop_cols = [
    'ticker',
    'fiscalDateEnding',
    'reportedCurrency_x',
    'reportedCurrency_y',
    'price_date',
    'Unnamed: 0'
]

df = df.drop(columns=drop_cols, errors='ignore')


In [5]:
X = df.drop(columns=['target'])
y = df['target']

In [6]:
split = int(len(df) * 0.8)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

In [7]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [9]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier( n_estimators=300,
                                max_depth=12,
                                random_state=42
                               )

model.fit(X_train, y_train)

RandomForestClassifier(max_depth=12, n_estimators=300, random_state=42)

In [10]:
pred = model.predict(X_test)

In [11]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

Accuracy: 0.5625
              precision    recall  f1-score   support

           0       0.71      0.31      0.43       146
           1       0.52      0.86      0.64       126

    accuracy                           0.56       272
   macro avg       0.62      0.58      0.54       272
weighted avg       0.62      0.56      0.53       272



In [12]:
y.value_counts()

target
1    730
0    630
Name: count, dtype: int64

In [13]:
import pandas as pd

pd.Series(pred).value_counts()

1    209
0     63
Name: count, dtype: int64

In [15]:
from xgboost import XGBClassifier

model = XGBClassifier( n_estimators=500, learning_rate=0.03,
                       max_depth=6, subsample=0.8,
                      colsample_bytree=0.8, random_state=42 )

model.fit(X_train, y_train)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.03, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=500, n_jobs=None,
              num_parallel_tree=None, random_state=42, ...)

In [16]:
from sklearn.metrics import roc_auc_score

proba = model.predict_proba(X_test)[:,1]

roc_auc_score(y_test, proba)

0.5996412263535551

In [17]:
proba = model.predict_proba(X_test)[:,1]

pred = (proba > 0.6).astype(int)

In [18]:
import numpy as np
from sklearn.metrics import accuracy_score

for t in np.arange(0.4,0.7,0.05):
    pred = (proba > t).astype(int)
    print(t, accuracy_score(y_test, pred))

0.4 0.5735294117647058
0.45 0.5625
0.5 0.5735294117647058
0.55 0.5845588235294118
0.6 0.5882352941176471
0.6499999999999999 0.5588235294117647


In [19]:
long_signal = proba > 0.65
short_signal = proba < 0.35

In [20]:
from lightgbm import LGBMClassifier

model = LGBMClassifier(
    n_estimators=600,
    learning_rate=0.03,
    max_depth=6
)

model.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 604, number of negative: 484
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001511 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11787
[LightGBM] [Info] Number of data points in the train set: 1088, number of used features: 65
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.555147 -> initscore=0.221489
[LightGBM] [Info] Start training from score 0.221489
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, bes

LGBMClassifier(learning_rate=0.03, max_depth=6, n_estimators=600)

In [21]:
from sklearn.metrics import roc_auc_score

proba = model.predict_proba(X_test)[:,1]

roc_auc_score(y_test, proba)

C:\Users\radha\AppData\Roaming\Python\Python39\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


0.6326918895412046